# Customer Churn Prediction — End to End (Colab Version)

Runs the full pipeline: upload data → clean → EDA → feature engineer → train models → evaluate → SHAP explainability → try live predictions.

**How to use:**
1. Run the cells top to bottom (Shift+Enter, or Runtime > Run all)
2. When prompted, upload your `telco_churn.csv` file
3. Everything else runs automatically


## 1. Install & import packages

In [ ]:
!pip install -q shap imbalanced-learn xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("All packages loaded.")

## 2. Upload the dataset
Download it first from [Kaggle: Telco Customer Churn](https://www.kaggle.com/blastchar/telco-customer-churn), then upload the CSV below.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f"Loaded: {csv_filename}")

## 3. Load & clean data

In [ ]:
df = pd.read_csv(csv_filename)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print(df.shape)
df.head()

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="Churn")
plt.title("Churn Distribution (0 = Stayed, 1 = Churned)")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure vs Churn")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 5. Feature engineering

In [ ]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 60, np.inf],
    labels=["0-1yr", "1-2yr", "2-4yr", "4-5yr", "5yr+"]
)

df["avg_monthly_spend"] = df["TotalCharges"] / (df["tenure"].replace(0, 1))

df.head()

## 6. Encode categorical features

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

X = df.drop(columns=["Churn"])
y = df["Churn"]
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

## 7. Train/test split, scaling, and SMOTE balancing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_train_bal).value_counts().to_dict()}")

## 8. Train models

In [ ]:
models = {}

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_bal, y_train_bal)
models["Logistic Regression"] = lr

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train_bal, y_train_bal)
models["Random Forest"] = rf

xgb_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    eval_metric="logloss", random_state=RANDOM_STATE
)
xgb_clf.fit(X_train_bal, y_train_bal)
models["XGBoost"] = xgb_clf

print("All 3 models trained.")

## 9. Evaluate models

In [ ]:
results = {}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"]))
    print(f"ROC-AUC: {auc:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    results[name] = {"model": model, "auc": auc, "f1": f1}

best_name = max(results, key=lambda k: results[k]["auc"])
best_model = results[best_name]["model"]
print(f"\nBest model: {best_name} (ROC-AUC: {results[best_name]['auc']:.4f})")

## 10. Explainability with SHAP
Shows which features push predictions toward churn vs. staying.

In [ ]:
explainer = shap.Explainer(best_model, X_train_bal)
shap_values = explainer(X_test_scaled)

shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns.tolist())

## 11. Try a live prediction
Edit the values below to simulate a customer and see the churn probability.

In [ ]:
sample_customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 5,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.0,
    "TotalCharges": 475.0,
}
sample_customer["tenure_group"] = "0-1yr"
sample_customer["avg_monthly_spend"] = sample_customer["TotalCharges"] / max(sample_customer["tenure"], 1)

input_df = pd.DataFrame([sample_customer])

for col, encoder in encoders.items():
    if col in input_df.columns:
        try:
            input_df[col] = encoder.transform(input_df[col].astype(str))
        except ValueError:
            input_df[col] = 0

input_df = input_df.reindex(columns=X.columns, fill_value=0)
input_scaled = scaler.transform(input_df)

prob = best_model.predict_proba(input_scaled)[0][1]
pred = best_model.predict(input_scaled)[0]

print(f"Churn probability: {prob*100:.1f}%")
print("Prediction:", "CHURN" if pred == 1 else "STAY")

## 12. Save model artifacts (optional — downloads to your computer)

In [ ]:
import joblib
from google.colab import files as colab_files

joblib.dump(best_model, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(encoders, "encoders.pkl")
joblib.dump(X.columns.tolist(), "feature_names.pkl")

colab_files.download("best_model.pkl")
colab_files.download("scaler.pkl")
colab_files.download("encoders.pkl")
colab_files.download("feature_names.pkl")